In [30]:
import re
import math
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [31]:
sectors = [
    "materials",
    "communication-services",
    "consumer-staples",
    "consumer-discretionary",
    "energy",
    "financials",
    "healthcare",
    "industrials",
    "real-estate",
    "technology",
    "utilities"
]


In [7]:
    
def scrape_sector(sector_name):
    """
    Scrape StockAnalysis sector page.

    Output columns:
    ticker, company_name, sector_name

    Example:
        scrape_sector("healthcare")
        scrape_sector("technology")
    """

    url = f"https://stockanalysis.com/stocks/sector/{sector_name}/"

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    r = requests.get(url, headers=headers, timeout=20)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    rows = []

    # Find ticker cells
    ticker_cells = soup.find_all("td", class_="sym")

    for ticker_cell in ticker_cells:

        ticker_tag = ticker_cell.find("a")
        if not ticker_tag:
            continue

        ticker = ticker_tag.get_text(strip=True)

        # company name is cell to the RIGHT with class="slw"
        company_cell = ticker_cell.find_next_sibling("td", class_="slw")

        if company_cell:
            company_name = company_cell.get_text(strip=True)
        else:
            company_name = ""

        rows.append({
            "ticker": ticker,
            "company_name": company_name,
            "sector_name": sector_name
        })

    df = pd.DataFrame(rows)

    return df



for sector in sectors:
    df = scrape_sector(sector)
    
    print(f"\nTotal Stocks Found: {len(df)}")
    print(df.head(20))
    
    df.to_csv(f"{sector}.csv", index=False)




Total Stocks Found: 278
   ticker                      company_name sector_name
0     LIN                         Linde plc   materials
1     BHP                 BHP Group Limited   materials
2     RIO                   Rio Tinto Group   materials
3    SCCO       Southern Copper Corporation   materials
4     NEM               Newmont Corporation   materials
5     AEM        Agnico Eagle Mines Limited   materials
6     FCX             Freeport-McMoRan Inc.   materials
7     SHW      The Sherwin-Williams Company   materials
8     CRH                           CRH plc   materials
9     ECL                       Ecolab Inc.   materials
10   VALE                         Vale S.A.   materials
11    APD  Air Products and Chemicals, Inc.   materials
12      B        Barrick Mining Corporation   materials
13    WPM     Wheaton Precious Metals Corp.   materials
14   CTVA                     Corteva, Inc.   materials
15    NUE                 Nucor Corporation   materials
16     AU             A

In [64]:
import pandas as pd
import glob
import os


def merge_sector_csvs(folder_path="."):
    """
    Merge all sector CSV files into one master CSV.

    Expected files:
        healthcare.csv
        technology.csv
        energy.csv
        etc.

    Output:
        all_stocks.csv
    """

    # Find all CSV files in folder
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

    # Optional: exclude previously merged file
    csv_files = [
        f for f in csv_files
        if os.path.basename(f) != "sector_stocks.csv"
    ]

    if not csv_files:
        print("No CSV files found.")
        return

    dfs = []

    for file in csv_files:
        print(f"Loading {file}")
        df = pd.read_csv(file)
        dfs.append(df)

    # Merge all files
    merged_df = pd.concat(dfs, ignore_index=True)

    # Remove duplicates if any
    merged_df.drop_duplicates(subset=["ticker"], inplace=True)

    # Save final file
    output_file = os.path.join(folder_path, "all_stocks.csv")
    merged_df.to_csv(output_file, index=False)

    print(f"\nMerged {len(csv_files)} files")
    print(f"Total Stocks: {len(merged_df)}")
    print(f"Saved to {output_file}")


# ---------------------------------------
# Run
# ---------------------------------------
merge_sector_csvs("C:/Udemy/2026-Python-EricBanas/Banas/Section53_Finance/redo/sector-stocks")

Loading C:/Udemy/2026-Python-EricBanas/Banas/Section53_Finance/redo/sector-stocks\communication-services.csv
Loading C:/Udemy/2026-Python-EricBanas/Banas/Section53_Finance/redo/sector-stocks\consumer-discretionary.csv
Loading C:/Udemy/2026-Python-EricBanas/Banas/Section53_Finance/redo/sector-stocks\consumer-staples.csv
Loading C:/Udemy/2026-Python-EricBanas/Banas/Section53_Finance/redo/sector-stocks\energy.csv
Loading C:/Udemy/2026-Python-EricBanas/Banas/Section53_Finance/redo/sector-stocks\financials.csv
Loading C:/Udemy/2026-Python-EricBanas/Banas/Section53_Finance/redo/sector-stocks\healthcare.csv
Loading C:/Udemy/2026-Python-EricBanas/Banas/Section53_Finance/redo/sector-stocks\industrials.csv
Loading C:/Udemy/2026-Python-EricBanas/Banas/Section53_Finance/redo/sector-stocks\materials.csv
Loading C:/Udemy/2026-Python-EricBanas/Banas/Section53_Finance/redo/sector-stocks\real-estate.csv
Loading C:/Udemy/2026-Python-EricBanas/Banas/Section53_Finance/redo/sector-stocks\technology.csv
Loa

In [63]:


HEADERS = {
    "User-Agent": "Mozilla/5.0"
}


# -------------------------------------------------------
# Get BeautifulSoup object
# -------------------------------------------------------
def get_soup(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")


# -------------------------------------------------------
# Find total pages from nav bar
#
# Structure:
# nav
#   div
#   div
#      span
#      span   <-- contains "Page 1 of 2"
#   a
# -------------------------------------------------------



# -------------------------------------------------------
# Scrape one page
# -------------------------------------------------------
def scrape_page(url, sector_name):
    soup = get_soup(url)

    rows = []

    ticker_cells = soup.find_all("td", class_="sym")

    for ticker_cell in ticker_cells:

        a = ticker_cell.find("a")
        if not a:
            continue

        ticker = a.get_text(strip=True)

        company_cell = ticker_cell.find_next_sibling("td", class_="slw")
        company_name = company_cell.get_text(strip=True) if company_cell else ""

        rows.append({
            "ticker": ticker,
            "company_name": company_name,
            "sector_name": sector_name
        })

    return rows


# -------------------------------------------------------
# Main sector scraper (all pages)
# -------------------------------------------------------
def scrape_sector(sector_name):
    """
    Example:
        scrape_sector("industrials")
        scrape_sector("technology")
    """

    base_url = f"https://stockanalysis.com/stocks/sector/{sector_name}/"

    # page 1
    soup = get_soup(base_url)

    print (998)
    total_pages = get_total_pages(soup)
    print (999)

    print (total_pages)
    
    print(f"{sector_name}: {total_pages} pages found")

    all_rows = []

    # scrape page 1
    all_rows.extend(scrape_page(base_url, sector_name))

    # scrape remaining pages
    for page_num in range(2, total_pages + 1):
        page_url = f"{base_url}?p={page_num}"
        print(f"Scraping page {page_num}...")

        rows = scrape_page(page_url, sector_name)
        all_rows.extend(rows)

    df = pd.DataFrame(all_rows)

    # remove duplicates if any
    df.drop_duplicates(subset=["ticker"], inplace=True)

    # save csv
    file_name = f"{sector_name}.csv"
    df.to_csv(file_name, index=False)

    print(f"Saved {len(df)} stocks to {file_name}")

    return df


def get_total_pages(soup):
    
    nav = soup.find_all("nav")
    
    if len(nav) == 2:
        return 1
        
    if not nav:
        return 1

    nav = nav[2]
    
    # Find the outer span that contains pagination text
    page_span = nav.find("span", class_="whitespace-nowrap")

    if not page_span:
        return 1

    text = page_span.get_text(" ", strip=True)
    # Example result: "Page 1 of 2"

    match = re.search(r"of\s+(\d+)", text)

    if match:
        return int(match.group(1))

    return 1
# -------------------------------------------------------
# Example Usage
# -------------------------------------------------------



for sector in sectors:
    df = scrape_sector(sector)
    print(df.head(20))
    
#     print(f"\nTotal Stocks Found: {len(df)}")
#     print(df.head(20))
    
#     df.to_csv(f"{sector}.csv", index=False)

# print(df.head(20))

998
999
1
materials: 1 pages found
Saved 278 stocks to materials.csv
   ticker                      company_name sector_name
0     LIN                         Linde plc   materials
1     BHP                 BHP Group Limited   materials
2     RIO                   Rio Tinto Group   materials
3    SCCO       Southern Copper Corporation   materials
4     NEM               Newmont Corporation   materials
5     AEM        Agnico Eagle Mines Limited   materials
6     FCX             Freeport-McMoRan Inc.   materials
7     SHW      The Sherwin-Williams Company   materials
8     CRH                           CRH plc   materials
9     ECL                       Ecolab Inc.   materials
10   VALE                         Vale S.A.   materials
11    APD  Air Products and Chemicals, Inc.   materials
12      B        Barrick Mining Corporation   materials
13    WPM     Wheaton Precious Metals Corp.   materials
14   CTVA                     Corteva, Inc.   materials
15    NUE                 Nucor Cor

## Analyze Industrials stocks